# Stage 6: Evaluation and Report Figures

**Primary author:** Sahana
**Builds on:**
- `04_clustering.ipynb` (Victoria — Stage 4: unconstrained HDBSCAN and agglomerative results)
- `05_constrained_and_targeted.ipynb` (Victoria — Stage 5: constrained clustering and subset experiments)
- `01_data_cleaning.ipynb` (Victoria — Stage 1: verified indicators and wordplay labels)
- `02_embedding_generation.ipynb` / `03_dimensionality_reduction.ipynb` (Stages 2–3: embeddings)

**Prompt engineering:** Victoria
**AI assistance:** Claude (Anthropic)
**Environment:** Local or Colab (no GPU required)

---

**Research question:** *"What is the best summary of our clustering results for the final report?"*

> **Colab note:** The environment-detection block below mounts Google Drive automatically.
> No GPU is needed. If loading the `.npy` files causes a memory error, go to
> **Runtime → Change runtime type** and select **High-RAM**.

## Section 1: Setup and Data Preparation

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import adjusted_rand_score
import numpy as np

# sklearn clustering metrics are NOT imported here because this notebook
# does no new clustering — all metrics come from saved CSVs.

warnings.filterwarnings("ignore")
np.random.seed(42)

# Report-quality figure defaults.
# REPORT_DPI=200 is higher than NB 04/05 (150) to support print quality.
REPORT_DPI = 200
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "figure.facecolor": "white",
})

print("Imports complete.")

### Environment Auto-Detection and Paths

In [ ]:
# --- Environment Auto-Detection ---
try:
    IS_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IS_COLAB = False

IS_GREATLAKES = "SLURM_JOB_ID" in os.environ

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/SIADS 692 Milestone II/Milestone II - NLP Cryptic Crossword Clues")
elif IS_GREATLAKES:
    PROJECT_ROOT = Path("/home/sahanasu/ccc-project/indicator_clustering")
else:
    try:
        PROJECT_ROOT = Path(__file__).resolve().parent.parent
    except NameError:
        PROJECT_ROOT = Path.cwd().parent

DATA_DIR    = PROJECT_ROOT / "data"
OUTPUT_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
REPORT_FIGURES_DIR = FIGURES_DIR / "report"  # Report-quality figures saved here

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:   {PROJECT_ROOT}")
print(f"Data dir:       {DATA_DIR}")
print(f"Output dir:     {OUTPUT_DIR}")
print(f"Report figs:    {REPORT_FIGURES_DIR}")

### Input File Validation

This notebook requires outputs from Stages 1–4. We check that all required files exist
before proceeding, rather than failing partway through.

| File | Produced by | Description |
|------|-------------|-------------|
| `embeddings_umap_10d.npy` | Stage 3 | 10D UMAP embeddings for clustering |
| `embeddings_umap_2d.npy` | Stage 3 | 2D UMAP embeddings for visualization |
| `indicator_index_all.csv` | Stage 2 | Row-to-indicator-string mapping |
| `verified_clues_labeled.csv` | Stage 1 | Clue-indicator pairs with Ho and GT labels |
| `cluster_labels_hdbscan_eps_0p0000.csv` | Stage 4 | Best HDBSCAN labels (eps=0.0) |
| `cluster_labels_agglo_k8.csv` | Stage 4 | Agglomerative k=8 labels |
| `cluster_labels_agglo_k10.csv` | Stage 4 | Agglomerative k=10 labels |
| `cluster_labels_agglo_k34.csv` | Stage 4 | Agglomerative k=34 labels |
| `clustering_metrics_summary.csv` | Stage 4 | Metrics from all Stage 4 runs |
| `wordplay_seeds.xlsx` | Manual (expert) | Seed words for constrained clustering (Section 3) |

**New in this notebook:** `verified_clues_labeled.csv` and `wordplay_seeds.xlsx`. These
were deliberately excluded from Notebook 04 to keep the unconstrained analysis label-free.
This is where domain knowledge enters the clustering pipeline for the first time.

In [ ]:
# Files required in DATA_DIR
required_files = {
    # Stage 2-3: embeddings
    "embeddings_umap_2d.npy":              "Run 03_dimensionality_reduction.ipynb (Stage 3)",
    "indicator_index_all.csv":              "Run 02_embedding_generation.ipynb (Stage 2)",
    # Stage 1: labels
    "verified_clues_labeled.csv":           "Run 01_data_cleaning.ipynb (Stage 1)",
    # Stage 4: cluster labels
    "cluster_labels_hdbscan_eps_0p0000.csv":"Run 04_clustering.ipynb (Stage 4)",
    "cluster_labels_agglo_k8.csv":          "Run 04_clustering.ipynb (Stage 4)",
    "cluster_labels_agglo_k10.csv":         "Run 04_clustering.ipynb (Stage 4)",
    "cluster_labels_agglo_k34.csv":         "Run 04_clustering.ipynb (Stage 4)",
    # Stage 5: constrained labels
    "cluster_labels_constrained_mc7.csv":   "Run 05_constrained_and_targeted.ipynb (Stage 5)",
    "cluster_labels_constrained_cg34.csv":  "Run 05_constrained_and_targeted.ipynb (Stage 5)",
}

# Files required in OUTPUT_DIR
required_output_files = {
    "clustering_metrics_summary.csv":           "Run 04_clustering.ipynb (Stage 4)",
    "constrained_vs_unconstrained_metrics.csv": "Run 05_constrained_and_targeted.ipynb (Stage 5)",
    "subset_experiment_metrics.csv":            "Run 05_constrained_and_targeted.ipynb (Stage 5)",
}

# Optional files — sections that need these will skip gracefully if absent
optional_files = {
    DATA_DIR / "embeddings_umap_10d.npy":                  "Section 4 anagram centroid analysis",
    DATA_DIR / "cluster_labels_subset_easy_agglo_k2.csv":  "Section 3 subset scatter (4A)",
    DATA_DIR / "cluster_labels_subset_hard_agglo_k3.csv":  "Section 3 subset scatter (4B)",
    DATA_DIR / "cluster_labels_subset_anagram_hdbscan.csv":"Section 4 anagram sub-clusters",
}

missing = []
for fname, fix_msg in required_files.items():
    fpath = DATA_DIR / fname
    if not fpath.exists():
        missing.append(f"  MISSING: {fpath}\n    Fix: {fix_msg}")
for fname, fix_msg in required_output_files.items():
    fpath = OUTPUT_DIR / fname
    if not fpath.exists():
        missing.append(f"  MISSING: {fpath}\n    Fix: {fix_msg}")

if missing:
    for m in missing:
        print(m)
    raise FileNotFoundError(
        f"{len(missing)} required file(s) missing. "
        "Generate them by running the notebooks listed above."
    )
print("All required files found.")

print("\nOptional files:")
for p, use in optional_files.items():
    status = "found" if p.exists() else "not found (will skip)"
    print(f"  [{status}] {p.name}  →  {use}")

In [ ]:
# 2D UMAP embeddings — used for all scatter plots in this notebook
embeddings_2d = np.load(DATA_DIR / "embeddings_umap_2d.npy")

# 10D UMAP embeddings — optional; needed only for centroid computation (Section 4)
emb10d_path = DATA_DIR / "embeddings_umap_10d.npy"
if emb10d_path.exists():
    embeddings_10d = np.load(emb10d_path)
    print(f"10D embeddings: {embeddings_10d.shape}")
else:
    embeddings_10d = None
    print("10D embeddings: not available (Section 4 centroid analysis will be skipped)")

# Indicator index: row i in embeddings_2d -> indicator string
df_index       = pd.read_csv(DATA_DIR / "indicator_index_all.csv", index_col=0)
indicator_names = df_index["indicator"].values

# Reverse map: indicator string -> embedding row index (used to align subset labels)
indicator_to_idx = {ind: i for i, ind in enumerate(indicator_names)}

print(f"2D embeddings:   {embeddings_2d.shape}  (expected: (12622, 2))")
print(f"Indicator count: {len(indicator_names):,}")
print(f"Sample: {indicator_names[:5]}")

### Load Wordplay Labels

`df_master` is built here using the same logic as **NB05 Section 1** — see that notebook for the full explanation of Ho labels, GT labels, and multi-label treatment. The key columns produced are `ho_labels` (frozenset of all Ho types per indicator), `gt_labels`, `primary_ho` (most frequent Ho type, used for colouring), and `primary_gt`.


In [ ]:
# Load the full labels file (same as NB05)
df_labels = pd.read_csv(DATA_DIR / 'verified_clues_labeled.csv')
print(f'Loaded verified_clues_labeled.csv: {len(df_labels):,} rows, '
      f'{df_labels["indicator"].nunique():,} unique indicators')


In [ ]:
# --- Build df_master: one row per indicator, aligned with embedding row order ---
# (Same construction as NB05; repeated here so NB06 is self-contained to run.)

ho_label_sets = (
    df_labels
    .groupby('indicator')['wordplay_ho']
    .apply(lambda x: frozenset(x.dropna().unique()))
    .to_dict()
)
gt_label_sets = (
    df_labels
    .groupby('indicator')['wordplay_gt']
    .apply(lambda x: frozenset(x.dropna().unique()))
    .to_dict()
)
primary_ho_map = (
    df_labels
    .groupby('indicator')['wordplay_ho']
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)
primary_gt_map = (
    df_labels[df_labels['wordplay_gt'].notna()]
    .groupby('indicator')['wordplay_gt']
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

df_master = df_index[['indicator']].copy()
df_master['ho_labels']  = df_master['indicator'].map(ho_label_sets)
df_master['gt_labels']  = df_master['indicator'].map(gt_label_sets)
df_master['primary_ho'] = df_master['indicator'].map(primary_ho_map)
df_master['primary_gt'] = df_master['indicator'].map(primary_gt_map)

n_no_ho = df_master['primary_ho'].isna().sum()
if n_no_ho > 0:
    print(f'Warning: {n_no_ho} indicators have no Ho label')
print(f'df_master ready: {len(df_master):,} rows')


### Load Cluster Label CSVs

We load all cluster label files saved by NB 04 (unconstrained runs) and NB 05
(constrained runs). Each CSV has two columns: `indicator` and `cluster_label`.
A helper function aligns every array to `indicator_names` (the canonical embedding
row order) so that `labels[i]` always corresponds to `embeddings_2d[i]`.

**HDBSCAN convention:** `cluster_label == -1` means the indicator was classified
as **noise** and excluded from cluster-quality scoring in NB 04. Noise points are
excluded from purity and silhouette computations here too.

In [ ]:
def load_cluster_csv(fpath):
    """Load a cluster-label CSV and return a numpy array aligned with indicator_names.

    Any indicator present in indicator_names but absent from the CSV receives -1.
    """
    df_tmp = pd.read_csv(fpath)
    label_series = df_tmp.set_index("indicator")["cluster_label"]
    return np.array([label_series.get(ind, -1) for ind in indicator_names])


# --- NB 04: all HDBSCAN epsilon variants ---
hdbscan_files = sorted(DATA_DIR.glob("cluster_labels_hdbscan_eps_*.csv"))
hdbscan_labels_dict = {}
for fpath in hdbscan_files:
    eps_str = fpath.stem.replace("cluster_labels_hdbscan_eps_", "")
    eps = float(eps_str.replace("p", "."))
    hdbscan_labels_dict[eps] = load_cluster_csv(fpath)

# --- NB 04: all agglomerative k variants ---
agglo_files = sorted(
    DATA_DIR.glob("cluster_labels_agglo_k*.csv"),
    key=lambda p: int(p.stem.split("_k")[-1])
)
agglo_labels_dict = {}
for fpath in agglo_files:
    k = int(fpath.stem.split("_k")[-1])
    agglo_labels_dict[k] = load_cluster_csv(fpath)

# --- NB 05: constrained labels (full 12 622-indicator runs) ---
labels_mc7  = load_cluster_csv(DATA_DIR / "cluster_labels_constrained_mc7.csv")
labels_cg34 = load_cluster_csv(DATA_DIR / "cluster_labels_constrained_cg34.csv")

# --- NB 05: subset experiment labels (partial datasets; kept as DataFrames) ---
subset_label_files = {
    "easy_agglo_k2":   "cluster_labels_subset_easy_agglo_k2.csv",
    "easy_hdbscan":    "cluster_labels_subset_easy_hdbscan.csv",
    "hard_agglo_k3":   "cluster_labels_subset_hard_agglo_k3.csv",
    "hard_hdbscan":    "cluster_labels_subset_hard_hdbscan.csv",
    "anagram_hdbscan": "cluster_labels_subset_anagram_hdbscan.csv",
    "anagram_agglo_k8":"cluster_labels_subset_anagram_agglo_k8.csv",
}
subset_labels = {}
for key, fname in subset_label_files.items():
    fpath = DATA_DIR / fname
    if fpath.exists():
        df_sub = pd.read_csv(fpath)
        df_sub["emb_idx"] = df_sub["indicator"].map(indicator_to_idx)
        subset_labels[key] = df_sub

print(f"HDBSCAN runs:      {len(hdbscan_labels_dict)}")
print(f"Agglomerative runs:{len(agglo_labels_dict)}")
print(f"Constrained MC7:   {len(set(labels_mc7[labels_mc7 != -1]))} clusters")
print(f"Constrained CG34:  {len(set(labels_cg34[labels_cg34 != -1]))} clusters")
print(f"Subset files:      {len(subset_labels)} of {len(subset_label_files)} found")

### Load Metrics CSVs

Three metrics files from NB 04 and NB 05 are loaded separately for section-specific
use, then combined into a single `df_all_metrics` dataframe for cross-method comparison.

| File | Source | Key columns |
|------|--------|-------------|
| `clustering_metrics_summary.csv` | NB 04 | method, epsilon, k, n\_clusters, noise\_pct, silhouette, davies\_bouldin, calinski\_harabasz |
| `constrained_vs_unconstrained_metrics.csv` | NB 05 | run, k, seeds, silhouette, davies\_bouldin, avg\_purity |
| `subset_experiment_metrics.csv` | NB 05 | experiment, method, k, ari\_vs\_ho, silhouette |

In [ ]:
# --- Load each metrics CSV (kept separate for section-specific use) ---
df_metrics        = pd.read_csv(OUTPUT_DIR / "clustering_metrics_summary.csv")
df_constrained    = pd.read_csv(OUTPUT_DIR / "constrained_vs_unconstrained_metrics.csv")
df_subset_metrics = pd.read_csv(OUTPUT_DIR / "subset_experiment_metrics.csv")

# Create the 'k' column
# Extracts digits after 'k=' if method is Agglomerative, else assigns NaN
df_metrics['k'] = np.where(
    df_metrics['method'] == 'Agglomerative (Ward)',
    df_metrics['parameters'].str.extract(r'k=(\d+)', expand=False).astype(float),
    np.nan
)

# Create the 'epsilon' column
# Extracts decimals/digits after 'eps=' if method is HDBSCAN, else assigns NaN
df_metrics['epsilon'] = np.where(
    df_metrics['method'] == 'HDBSCAN',
    df_metrics['parameters'].str.extract(r'eps=([0-9.]+)', expand=False).astype(float),
    np.nan
)

# Optional: pre-computed type-distribution tables from NB 05
type_distributions = {}
for fpath in sorted(OUTPUT_DIR.glob("type_distribution_*.csv")):
    key = fpath.stem.replace("type_distribution_", "")
    type_distributions[key] = pd.read_csv(fpath, index_col=0)

# --- Build unified metrics dataframe for cross-method comparison ---
# Common schema: method_label, method_type, k, silhouette, davies_bouldin,
# avg_purity, ari_vs_ho, notes. Columns not applicable to a run are NaN.
unified_rows = []

for _, row in df_metrics.iterrows():
    if row["method"] == "hdbscan":
        label = f"HDBSCAN ε={row['epsilon']:.4f}"
        notes = f"{int(row.get('n_clusters', 0))} clusters, {row.get('noise_pct', 0):.1f}% noise"
    else:
        label = f"Agglomerative k={int(row['n_clusters'])}"
        notes = ""
    unified_rows.append({
        "method_label":  label,
        "method_type":   row["method"],
        "k":             row.get("k") if row["method"] == "Agglomerative (Ward)" else row.get("n_clusters"), 
        "silhouette":    row.get("silhouette"),
        "davies_bouldin":row.get("davies_bouldin"),
        "avg_purity":    None,
        "ari_vs_ho":     None,
        "notes":         notes,
    })

df_all_metrics = pd.DataFrame(unified_rows)

print(f"clustering_metrics_summary:           {len(df_metrics)} rows")
print(f"constrained_vs_unconstrained_metrics: {len(df_constrained)} rows")
print(f"subset_experiment_metrics:            {len(df_subset_metrics)} rows")
print(f"type_distribution tables:             {len(type_distributions)}")
print(f"df_all_metrics (unified):             {len(df_all_metrics)} rows")

In [ ]:
SEP = "=" * 60
print(SEP)
print("SECTION 1 SUMMARY")
print(SEP)
print(f"  Unique indicators:        {len(indicator_names):>7,}")
print(f"  UMAP 2D embeddings:       {str(embeddings_2d.shape):>14}")
emb10d_str = str(embeddings_10d.shape) if embeddings_10d is not None else "not loaded"
print(f"  UMAP 10D embeddings:      {emb10d_str:>14}")
print(f"  df_master rows:           {len(df_master):>7,}")
print()
print(f"  HDBSCAN label arrays:     {len(hdbscan_labels_dict):>7}")
print(f"  Agglomerative arrays:     {len(agglo_labels_dict):>7}")
print(f"  Constrained arrays:       {2:>7}  (MC7 k=7, CG34 k=34)")
print(f"  Subset label DataFrames:  {len(subset_labels):>7}")
print()
print(f"  NB04 metrics rows:        {len(df_metrics):>7}")
print(f"  Constrained metrics rows: {len(df_constrained):>7}")
print(f"  Subset metrics rows:      {len(df_subset_metrics):>7}")
print(f"  df_all_metrics (unified): {len(df_all_metrics):>7} rows")
print()
print(f"  Report figures → {REPORT_FIGURES_DIR}")
print(SEP)
print("Section 1 complete. Ready for Sections 2-4.")

---
## Section 2: Report-Quality Cross-Method Metrics Comparison

This section produces the figures for the report's evaluation section.
No new clustering is performed; all values come from the CSVs and label arrays
loaded in Section 1.

**Figures produced here:**
- **Figure 1** — Unified metrics comparison: all HDBSCAN runs, all agglomerative k
  runs, and both constrained runs on silhouette, Davies-Bouldin, and average cluster
  purity. Satisfies the rubric requirement for *"at least one overall summary
  comparing the best model from each family."*
- **Figure 2** — Sensitivity analysis (2-panel): HDBSCAN epsilon sweep (clusters
  found + silhouette vs ε) and agglomerative k sweep (silhouette + Davies-Bouldin
  vs k). Satisfies the rubric requirement for *sensitivity analysis on the best
  model from each family.*


### 2.1 Color Palette and Type Masks

We define the wordplay-type color palette (identical to NB 05 for visual consistency)
and pre-compute boolean membership masks. The masks are needed here because
**average cluster purity** — one of the three metrics in Figure 1 — is computed
directly from cluster label arrays and Ho type membership.

**Average cluster purity:** for each cluster, the fraction of its members whose
primary Ho label matches the cluster's most common type. Purity = mean of those
fractions across all clusters (noise points excluded).

A key reference value is the **anagram baseline**: if every indicator were in one
cluster, purity would equal the anagram fraction (~0.51). Any run whose purity is
below this is not separating types better than the trivial "everything is anagram"
assignment.

In [ ]:
HO_TYPES = ["anagram", "reversal", "hidden", "container",
            "insertion", "deletion", "homophone", "alternation"]
GT_TYPES = ["anagram", "reversal", "hidden", "alternation"]

# Per-indicator type color palette — identical to NB 05
TYPE_COLORS = {
    "anagram":     "#e41a1c",
    "reversal":    "#377eb8",
    "hidden":      "#4daf4a",
    "container":   "#984ea3",
    "insertion":   "#ff7f00",
    "deletion":    "#a65628",
    "homophone":   "#f781bf",
    "alternation": "#999999",
}

# Method-family colors for cross-method figures
FAMILY_COLORS = {
    "HDBSCAN":       "#2166ac",
    "Agglomerative": "#d6604d",
    "Constrained":   "#4dac26",
}

# Boolean membership masks: ho_type_masks[t][i] is True iff indicator i
# appears under type t in verified_clues_labeled.csv.
# Multi-label indicators (e.g., "about" = container + reversal + anagram)
# are True in multiple masks simultaneously.
ho_type_masks = {
    wtype: np.array([
        wtype in s if isinstance(s, frozenset) else False
        for s in df_master["ho_labels"]
    ])
    for wtype in HO_TYPES
}

gt_type_masks = {
    wtype: np.array([
        wtype in s if isinstance(s, frozenset) else False
        for s in df_master["gt_labels"]
    ])
    for wtype in GT_TYPES
}

# Anagram baseline: purity of a single cluster containing all indicators
ANAGRAM_BASELINE = float((df_master["primary_ho"] == "anagram").mean())

print(f"Anagram baseline purity: {ANAGRAM_BASELINE:.3f}")
print("Ho type counts:")
for wtype in HO_TYPES:
    print(f"  {wtype:12s}: {ho_type_masks[wtype].sum():5,}")

### 2.2 Unified Metrics Comparison (All Runs)

This figure includes **every clustering run** from NB 04 and NB 05.

| Metric | Better direction | What it measures |
|--------|-----------------|------------------|
| Silhouette score | Higher | How well each point fits its own cluster vs the nearest other cluster |
| Davies-Bouldin index | Lower | Average ratio of within-cluster scatter to between-cluster separation |
| Avg cluster purity | Higher | Mean fraction of each cluster's dominant Ho wordplay type |

Runs are **grouped by method family** with a separator line between groups.
Key configurations are marked with ★ and drawn at full opacity; all other runs
are dimmed to 50% so the highlights stand out without hiding the sweep context.

Reference lines:
- **Silhouette = 0** — below this the average cluster is less cohesive than random
- **Anagram baseline** on purity — below this the clustering is not separating types
  better than a single "everyone is anagram" cluster

In [ ]:
def compute_avg_purity(labels):
    """Average cluster purity over Ho wordplay types.

    For each cluster, purity = count of its most common Ho type / cluster size.
    Returns the mean purity across all non-noise clusters.

    Note: purity naturally inflates at very high k because small clusters tend
    to be more homogeneous. Always interpret alongside cluster count.
    """
    cluster_ids = sorted(set(int(c) for c in labels if c != -1))
    if not cluster_ids:
        return np.nan
    purities = []
    for c in cluster_ids:
        mask = labels == c
        n = int(mask.sum())
        if n == 0:
            continue
        max_count = max(int(ho_type_masks[t][mask].sum()) for t in HO_TYPES)
        purities.append(max_count / n)
    return float(np.mean(purities))


# Compute for all runs
purity_hdbscan = {eps: compute_avg_purity(lbl)
                  for eps, lbl in hdbscan_labels_dict.items()}
purity_agglo   = {k:   compute_avg_purity(lbl)
                  for k,   lbl in agglo_labels_dict.items()}
purity_mc7     = compute_avg_purity(labels_mc7)
purity_cg34    = compute_avg_purity(labels_cg34)

print("Avg purity — key runs:")
print(f"  HDBSCAN eps=0.00:         {purity_hdbscan.get(0.0, float('nan')):.3f}")
print(f"  Agglomerative k=8:        {purity_agglo.get(8,  float('nan')):.3f}")
print(f"  Agglomerative k=10:       {purity_agglo.get(10, float('nan')):.3f}")
print(f"  Agglomerative k=34:       {purity_agglo.get(34, float('nan')):.3f}")
print(f"  Constrained MC7  (k=7):   {purity_mc7:.3f}")
print(f"  Constrained CG34 (k=34):  {purity_cg34:.3f}")

In [ ]:
def get_nb04_metric(method, col, k=None, epsilon=None):
    """Pull a single metric value from df_metrics for a given run."""
    df = df_metrics[df_metrics["method"] == method]
    if k       is not None: df = df[df["k"] == k]
    if epsilon is not None: df = df[(df["epsilon"] - epsilon).abs() < 1e-6]
    return float(df[col].values[0]) if len(df) > 0 else np.nan

def get_constrained_metric(search_str, col):
    """Pull a metric from df_constrained by partial run-name match."""
    run_col = next((c for c in ["run","Run","method"] if c in df_constrained.columns), None)
    if run_col is None:
        return np.nan
    mask = df_constrained[run_col].str.lower().str.contains(search_str.lower(), na=False)
    vals = df_constrained.loc[mask, col]
    return float(vals.values[0]) if len(vals) > 0 else np.nan


HIGHLIGHT_EPS = {0.0}        # Best HDBSCAN silhouette
HIGHLIGHT_K   = {8, 10, 34}  # Wordplay-type target / local optimum / conceptual granularity

rows = []

for eps in sorted(hdbscan_labels_dict.keys()):
    lbl = hdbscan_labels_dict[eps]
    rows.append({
        "run":            f"HDBSCAN  ε={eps:.4f}",
        "method_family":  "HDBSCAN",
        "k":              int(len(set(lbl[lbl != -1]))),
        "noise_pct":      float(100 * (lbl == -1).mean()),
        "silhouette":     get_nb04_metric("HDBSCAN",      "silhouette",     epsilon=eps),
        "davies_bouldin": get_nb04_metric("HDBSCAN",      "davies_bouldin", epsilon=eps),
        "avg_purity":     purity_hdbscan.get(eps, np.nan),
        "highlight":      eps in HIGHLIGHT_EPS,
    })

for k in sorted(agglo_labels_dict.keys()):
    rows.append({
        "run":            f"Agglo  k={k}",
        "method_family":  "Agglomerative",
        "k":              k,
        "noise_pct":      0.0,
        "silhouette":     get_nb04_metric("Agglomerative (Ward)", "silhouette",     k=k),
        "davies_bouldin": get_nb04_metric("Agglomerative (Ward)", "davies_bouldin", k=k),
        "avg_purity":     purity_agglo.get(k, np.nan),
        "highlight":      k in HIGHLIGHT_K,
    })

for label, search, k_val, purity in [
    ("Constrained MC7  (k=7)",  "MC7",  7,  purity_mc7),
    ("Constrained CG34 (k=34)", "CG34", 34, purity_cg34),
]:
    rows.append({
        "run":            label,
        "method_family":  "Constrained",
        "k":              k_val,
        "noise_pct":      0.0,
        "silhouette":     get_constrained_metric(search, "Silhouette"),
        "davies_bouldin": get_constrained_metric(search, "Davies-Bouldin"),
        "avg_purity":     purity,
        "highlight":      True,
    })

df_display = pd.DataFrame(rows)
print(f"df_display: {len(df_display)} rows")
print(df_display[["run","k","silhouette","davies_bouldin","avg_purity"]].to_string(index=False))

In [ ]:
n_runs    = len(df_display)
y_pos     = np.arange(n_runs)
bar_h     = 0.72

bar_colors = [FAMILY_COLORS[f] for f in df_display["method_family"]]
bar_alphas = [0.90 if h else 0.50 for h in df_display["highlight"]]

# y-tick labels: prepend ★ for highlighted runs
ytick_labels = [
    ("★ " if h else "    ") + lbl
    for lbl, h in zip(df_display["run"], df_display["highlight"])
]

fig_height = max(9, n_runs * 0.40 + 2.5)
fig, axes  = plt.subplots(1, 3, figsize=(15, fig_height), sharey=True)

metrics_cfg = [
    ("silhouette",     "Silhouette score\n(higher = better)",   True),
    ("davies_bouldin", "Davies-Bouldin index\n(lower = better)", False),
    ("avg_purity",     "Avg cluster purity\n(higher = better)", True),
]

for ax, (col, title, higher_better) in zip(axes, metrics_cfg):
    vals = df_display[col].values

    for i, (val, color, alpha) in enumerate(zip(vals, bar_colors, bar_alphas)):
        if not np.isnan(val):
            ax.barh(i, val, color=color, alpha=alpha,
                    edgecolor="white", linewidth=0.4, height=bar_h)
            # Data label at right edge of bar, inside axis
            ax.text(val, i, f" {val:.2f}",
                    va="center", ha="left", fontsize=6.5,
                    color="black")

    # Reference lines
    if col == "silhouette":
        ax.axvline(0, color="black", linewidth=0.9, linestyle="--", alpha=0.7,
                   label="Silhouette = 0")
        ax.legend(fontsize=7.5, loc="lower right")
    if col == "avg_purity":
        ax.axvline(ANAGRAM_BASELINE, color="crimson", linewidth=0.9,
                   linestyle="--", alpha=0.7,
                   label=f"Anagram baseline ({ANAGRAM_BASELINE:.2f})")
        ax.legend(fontsize=7.5, loc="lower right")

    # Method-family separator lines
    for i in range(1, n_runs):
        if df_display.iloc[i]["method_family"] != df_display.iloc[i - 1]["method_family"]:
            ax.axhline(i - 0.5, color="#444444", linewidth=1.0, linestyle="-")

    # Title at top; no x-axis label
    ax.set_title(title, fontsize=9, pad=6)
    ax.grid(True, axis="x", alpha=0.25)
    ax.set_xlim(left=0)

# Expand right side so data labels aren't clipped
for ax in axes:
    xlo, xhi = ax.get_xlim()
    ax.set_xlim(xlo, xhi * 1.12)

# Shared y-axis: labels only on leftmost panel
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(ytick_labels, fontsize=7.5, family="monospace")
axes[0].invert_yaxis()  # eps=0 / k=4 at top; constrained at bottom

# Method-family legend in centre panel
legend_patches = [
    mpatches.Patch(color=FAMILY_COLORS["HDBSCAN"],
                   label=f"HDBSCAN ({len(hdbscan_labels_dict)} ε values)"),
    mpatches.Patch(color=FAMILY_COLORS["Agglomerative"],
                   label=f"Agglomerative Ward's ({len(agglo_labels_dict)} k values)"),
    mpatches.Patch(color=FAMILY_COLORS["Constrained"],
                   label="Constrained seed-word (2 runs)"),
]
axes[1].legend(handles=legend_patches, loc="lower right",
               fontsize=8, framealpha=0.9,
               title="Method family  |  ★ = key run", title_fontsize=7.5)

fig.suptitle(
    "Unified Metrics Comparison Across All Clustering Runs\n"
    f"({len(hdbscan_labels_dict)} HDBSCAN ε values  ·  "
    f"{len(agglo_labels_dict)} agglomerative k values  ·  2 constrained runs)",
    fontsize=12, y=1.01
)

plt.tight_layout()
fig.savefig(REPORT_FIGURES_DIR / "fig1_unified_metrics_comparison.png",
            dpi=REPORT_DPI, bbox_inches="tight")
plt.show()
print("Saved: fig1_unified_metrics_comparison.png")

# Save underlying numbers for the report appendix
df_display.to_csv(OUTPUT_DIR / "fig1_unified_metrics_data.csv", index=False)
print("Saved: fig1_unified_metrics_data.csv")

### 2.3 Sensitivity Analysis

Understanding how sensitive each method is to its primary hyperparameter is a
key analysis requirement. This figure answers:
*"Across the full parameter sweep, where does clustering quality peak, and how
quickly does it degrade?"*

**Left panel — HDBSCAN epsilon (ε) sensitivity:**
- **Blue line (left axis, log scale):** Number of non-noise clusters found at each ε
- **Orange line (right axis):** Silhouette score at each ε

The HDBSCAN sweep spans 13 ε values from 0 to 2.68, with finer resolution in the
low-ε transition zone (where the most change occurs). The key finding is the
**abrupt transition**: at ε=0, HDBSCAN finds ~280 fine-grained clusters
(silhouette ≈ 0.63), but by ε ≈ 1.5 the data collapses to 3–4 coarse groups with
poor silhouette. There is no stable intermediate regime — no value of ε gives both
a moderate cluster count *and* a good silhouette score. This is the
"no stable middle ground" finding that is central to Stage 4's conclusions.

**Right panel — Agglomerative Ward's k sensitivity:**
- **Blue line (left axis):** Silhouette score across k = 4 to 250
- **Orange line (right axis):** Davies-Bouldin index (lower = better) across k

The agglomerative sweep spans 17 k values. The key finding is the **absence of an
elbow**: silhouette improves monotonically and Davies-Bouldin declines monotonically
up to k = 250, with no inflection point. This tells us the data's geometric
structure is naturally finer-grained than any theory-motivated granularity (k = 3,
k = 8, k = 14). Vertical dashed lines mark k = 8 (the Ho wordplay-type count) and
k = 10 (the local silhouette optimum identified in Stage 5 analysis).


In [ ]:
# ─── Gather HDBSCAN sensitivity data ────────────────────────────────────────
# n_clusters and noise_pct are computed directly from the label arrays
# (more reliable than reading from df_metrics, which may use different column names)
eps_arr = np.array(sorted(hdbscan_labels_dict.keys()))

n_clust_arr = np.array([
    len(set(int(c) for c in hdbscan_labels_dict[e] if c != -1))
    for e in eps_arr
])

# Silhouette is taken from the pre-computed NB04 metrics CSV
def _hdbscan_sil(eps):
    mask = (
        (df_metrics['method'] == 'HDBSCAN') &
        ((df_metrics['epsilon'] - eps).abs() < 1e-4)
    )
    vals = df_metrics.loc[mask, 'silhouette']
    return float(vals.values[0]) if len(vals) > 0 else np.nan

sil_hdbscan_arr = np.array([_hdbscan_sil(e) for e in eps_arr])

# ─── Gather Agglomerative sensitivity data ────────────────────────────────────
k_arr = np.array(sorted(agglo_labels_dict.keys()))

def _agglo_metric(k, col):
    mask = (
        (df_metrics['method'] == 'Agglomerative (Ward)') &
        (df_metrics['k'] == k)
    )
    vals = df_metrics.loc[mask, col]
    return float(vals.values[0]) if len(vals) > 0 else np.nan

sil_agglo_arr = np.array([_agglo_metric(k, 'silhouette')     for k in k_arr])
db_agglo_arr  = np.array([_agglo_metric(k, 'davies_bouldin') for k in k_arr])

print(f"HDBSCAN sweep: {len(eps_arr)} epsilon values")
print(f"  eps range: {eps_arr[0]:.4f} – {eps_arr[-1]:.4f}")
print(f"  cluster count range: {n_clust_arr.min()} – {n_clust_arr.max()}")
print(f"  silhouette range:    {np.nanmin(sil_hdbscan_arr):.3f} – {np.nanmax(sil_hdbscan_arr):.3f}")
print(f"Agglomerative sweep: {len(k_arr)} k values")
print(f"  k range: {k_arr[0]} – {k_arr[-1]}")
print(f"  silhouette range:    {np.nanmin(sil_agglo_arr):.3f} – {np.nanmax(sil_agglo_arr):.3f}")
print(f"  Davies-Bouldin range:{np.nanmin(db_agglo_arr):.3f} – {np.nanmax(db_agglo_arr):.3f}")


In [ ]:
# ─── 2-panel sensitivity analysis ─────────────────────────────────
COL_BLUE   = '#2166ac'   # clusters (left axis) / silhouette (both panels)
COL_ORANGE = '#d6604d'   # silhouette (HDBSCAN right) / Davies-Bouldin (agglo right)

fig, (ax_L, ax_R) = plt.subplots(1, 2, figsize=(14, 5))
fig.subplots_adjust(wspace=0.44)   # extra horizontal space for dual y-axis labels

# ── Left panel: HDBSCAN epsilon sensitivity ───────────────────────────────────
ax_L_r = ax_L.twinx()   # right axis: silhouette

l1, = ax_L.plot(
    eps_arr, n_clust_arr, 'o-',
    color=COL_BLUE, linewidth=2.0, markersize=5, label='# clusters'
)
l2, = ax_L_r.plot(
    eps_arr, sil_hdbscan_arr, 's--',
    color=COL_ORANGE, linewidth=2.0, markersize=5, label='Silhouette'
)

# Silhouette = 0 reference line (below this, clusters are less cohesive than random)
ax_L_r.axhline(0, color='black', linewidth=0.9, linestyle=':', alpha=0.55,
               label='Silhouette = 0')

# Mark the best-silhouette epsilon
best_eps_idx = int(np.nanargmax(sil_hdbscan_arr))
best_eps_val = float(eps_arr[best_eps_idx])
ax_L.axvline(best_eps_val, color='#555555', linewidth=1.1, linestyle='--', alpha=0.75)
# get_xaxis_transform(): x in data units, y in axes fraction [0, 1]
ax_L.text(
    best_eps_val + max(eps_arr) * 0.015, 0.93,
    f'\u03b5={best_eps_val:.2f}\n(best sil.)',
    transform=ax_L.get_xaxis_transform(),
    fontsize=7.5, color='#555555', va='top', ha='left'
)

# Log scale on cluster-count axis: counts span 1 – ~280
ax_L.set_yscale('log')
ax_L.set_xlabel('HDBSCAN epsilon (\u03b5)', fontsize=10)
ax_L.set_ylabel('# clusters  (log scale)', color=COL_BLUE, fontsize=10)
ax_L_r.set_ylabel('Silhouette score', color=COL_ORANGE, fontsize=10)
ax_L.tick_params(axis='y', labelcolor=COL_BLUE)
ax_L_r.tick_params(axis='y', labelcolor=COL_ORANGE)
ax_L.set_title('HDBSCAN: Clusters Found & Silhouette vs \u03b5', fontsize=11, pad=8)
ax_L.grid(True, alpha=0.20)

# Combined legend: show both lines
ax_L.legend(
    [l1, l2], [l1.get_label(), l2.get_label()],
    loc='center right', fontsize=8.5, framealpha=0.9
)

# ── Right panel: Agglomerative k sensitivity ───────────────────────────────────
ax_R_r = ax_R.twinx()   # right axis: Davies-Bouldin

l3, = ax_R.plot(
    k_arr, sil_agglo_arr, 'o-',
    color=COL_BLUE, linewidth=2.0, markersize=5, label='Silhouette'
)
l4, = ax_R_r.plot(
    k_arr, db_agglo_arr, 's--',
    color=COL_ORANGE, linewidth=2.0, markersize=5, label='Davies-Bouldin'
)

# Annotate key k values
for k_ref, k_lbl in [(8, 'k=8\n(Ho types)'), (10, 'k=10\n(local opt.)')]:
    if k_ref in agglo_labels_dict:
        ax_R.axvline(k_ref, color='#555555', linewidth=1.0, linestyle='--', alpha=0.70)
        # x offset: multiply by small factor (works correctly on log-scaled axis)
        ax_R.text(
            k_ref * 1.08, 0.06, k_lbl,
            transform=ax_R.get_xaxis_transform(),
            fontsize=7.5, color='#555555', va='bottom', ha='left'
        )

ax_R.set_xscale('log')   # k spans 4–250; log scale separates small-k points
ax_R.set_xlabel('Number of clusters (k)', fontsize=10)
ax_R.set_ylabel('Silhouette score', color=COL_BLUE, fontsize=10)
ax_R_r.set_ylabel('Davies-Bouldin index', color=COL_ORANGE, fontsize=10)
ax_R.tick_params(axis='y', labelcolor=COL_BLUE)
ax_R_r.tick_params(axis='y', labelcolor=COL_ORANGE)
ax_R.set_title("Agglomerative (Ward's): Silhouette & DB vs k", fontsize=11, pad=8)
ax_R.grid(True, alpha=0.20)

ax_R.legend(
    [l3, l4], [l3.get_label(), l4.get_label()],
    loc='upper left', fontsize=8.5, framealpha=0.9
)

# ── Overall title and save ─────────────────────────────────────────────────────
fig.suptitle(
    'Sensitivity Analysis\n'
    '(Left: HDBSCAN \u03b5 sweep  |  Right: Agglomerative Ward\u2019s k sweep)',
    fontsize=12, y=1.04
)

plt.tight_layout()
fig.savefig(
    REPORT_FIGURES_DIR / 'fig2_sensitivity_analysis.png',
    dpi=REPORT_DPI, bbox_inches='tight'
)
plt.show()
print('Saved: fig2_sensitivity_analysis.png')


---
## Section 3: Key Result Figures

Section 2 compared methods *numerically* (silhouette, Davies-Bouldin, purity).
Section 3 examines them *spatially and compositionally* — overlaying wordplay-type
labels onto the UMAP embedding and inspecting what lives inside each cluster.

Because the 2D embedding is a fixed coordinate system (computed once in Stage 3
and never changed), every scatter-plot panel is directly comparable: the same
point at the same (x, y) position represents the same indicator across all panels.

**Figures produced here:**
- **Figure 3** `3.1` — Ho type overlay (2×4 grid): spatial footprint of each of the
  8 Ho types. Establishes that types *do* have geometric structure in embedding space.
- **Figure 4** `3.2` — Easy vs. Hard separation (1×2 scatter): the 13.5× ARI contrast
  between Experiment 4A (homophone + reversal, ARI = 0.611) and 4B (hidden + container
  + insertion, ARI = 0.045). The single strongest quantitative finding.
- **Figure 5** `3.3` — Constrained vs. unconstrained heatmaps (1×2): per-cluster Ho
  type composition for agglo k=8 (unconstrained) vs. MC7 k=7 (constrained). Shows
  the marginal purity gain from injecting expert seed words.
- **Figure 6** `3.4` — Anagram sub-clustering (2×2 scatter): HDBSCAN, k=4, k=8, and
  k≈12 applied to anagram-only indicators, revealing rich conceptual-metaphor
  sub-structure within the dominant type.


### 3.1 Ho Type Overlay

**What this figure shows:** Each of the 8 panels plots all 12,622 indicators in
the same UMAP 2D space, but highlights a different Ho wordplay type. The gray
background shows the full embedding cloud; the colored overlay shows which
indicators belong to that type.

**Why this matters for the report:** This is the primary evidence for our claim
that wordplay types *do* have geometric structure in semantic embedding space —
some types occupy compact, well-defined regions while others are diffuse. If the
colored points were randomly scattered over the gray cloud, clustering by semantics
would be pointless. The degree of spatial concentration tells us how much signal
the embeddings carry for each type.

**Multi-label indicators:** An indicator like *"about"* that functions as both a
container and a reversal indicator will appear (colored) in both the container
panel and the reversal panel. This is intentional — `ho_type_masks` captures
all clue appearances, not just the primary label. It means the total colored
point count across all 8 panels exceeds 12,622.

**What to look for:**
- **Compact, dense colored region** → the embedding space successfully separates
  that type (good signal for clustering). *Expect: homophone, alternation.*
- **Diffuse or multi-modal colored region** → the type spans multiple semantic
  areas and will be hard to cluster purely. *Expect: anagram (largest type;*
  *internally sub-structured).*
- **Heavy overlap between types** → confirms the intermixing finding.
  *Expect: container + insertion.*

**Implementation note:** Point sizes and alpha values are deliberately tuned for
a 12k-point dataset at 300 DPI. The gray background uses `alpha=0.08` to avoid
overplotting; the colored overlay uses `alpha=0.45` so individual points remain
distinguishable. `rasterized=True` keeps the saved PNG at a manageable file size.


In [ ]:
# ─── Ho Type Overlay (2×4 grid) ────────────────────────────────────
# All data needed here was loaded in Section 1:
#   embeddings_2d  — shape (12622, 2), 2D UMAP coordinates
#   ho_type_masks  — dict[str, ndarray[bool]], built from verified_clues_labeled.csv
#   TYPE_COLORS    — dict[str, hex_str], defined in Section 2.1
#   HO_TYPES       — ordered list of 8 type names, defined in Section 2.1
# No embeddings are recomputed here.

x_all = embeddings_2d[:, 0]   # UMAP dimension 1
y_all = embeddings_2d[:, 1]   # UMAP dimension 2

# ── Layout ────────────────────────────────────────────────────────────────────
# 2 rows × 4 columns, one panel per Ho wordplay type (alphabetical order not
# used; HO_TYPES preserves the conventional ordering: anagram first)
FIG3_DPI = 300   # Higher than REPORT_DPI (200) — this is the primary visual
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes_flat = axes.flatten()   # iterate as a 1-D sequence

# Global scatter keyword arguments shared by all panels
BG_KW = dict(
    c='#c8c8c8',         # neutral gray — not white (invisible) or dark (distracting)
    s=0.7,               # small: with 12k points even tiny markers fill the space
    alpha=0.08,          # very transparent: prevents gray from dominating
    linewidths=0,        # no marker edges (avoids fringing at small sizes)
    rasterized=True,     # rasterise scatter to keep PNG file size manageable
)
FG_KW = dict(
    s=2.5,               # larger than background so highlighted points read clearly
    alpha=0.45,          # semi-transparent: shows density variation within the type
    linewidths=0,
    rasterized=True,
)

# ── Per-panel loop ────────────────────────────────────────────────────────────
for ax, wtype in zip(axes_flat, HO_TYPES):

    # Layer 1 — gray background: ALL 12,622 indicators
    # Plotting the full cloud first establishes the shape of the embedding space
    # so the reader can judge how much of it the highlighted type occupies.
    ax.scatter(x_all, y_all, **BG_KW)

    # Layer 2 — colored overlay: only indicators that belong to this type
    # ho_type_masks[wtype] is True for every indicator that appears under this
    # type in at least one clue in verified_clues_labeled.csv (multi-label aware).
    mask = ho_type_masks[wtype]
    n_type = int(mask.sum())
    ax.scatter(x_all[mask], y_all[mask], c=TYPE_COLORS[wtype], **FG_KW)

    # ── Panel title: type name + indicator count ──────────────────────────────
    # Color the title to match the overlay — creates an immediate visual link
    # between the legend information and the panel content.
    ax.set_title(
        f"{wtype}\n(n\u202f=\u202f{n_type:,})",   # narrow non-breaking space around =
        fontsize=10.5,
        fontweight='bold',
        color=TYPE_COLORS[wtype],
        pad=4,
    )

    # ── Axis aesthetics ───────────────────────────────────────────────────────
    # Suppress tick marks — the absolute coordinate values are meaningless
    # (UMAP is not metric-preserving). Only relative position matters.
    ax.set_xticks([])
    ax.set_yticks([])

    # Add axis dimension labels only on the edge panels to reduce clutter:
    #   bottom row  → show 'UMAP 1' on x-axis  (row index 1 of a 2-row grid)
    #   left column → show 'UMAP 2' on y-axis  (column index 0)
    panel_idx = list(HO_TYPES).index(wtype)
    row, col  = divmod(panel_idx, 4)
    if row == 1:   # bottom row
        ax.set_xlabel('UMAP 1', fontsize=8, labelpad=2)
    if col == 0:   # left column
        ax.set_ylabel('UMAP 2', fontsize=8, labelpad=2)

    # Thin gray border — keeps panels visually separated without heavy lines
    for spine in ax.spines.values():
        spine.set_edgecolor('#cccccc')
        spine.set_linewidth(0.6)

# ── Figure-level title ────────────────────────────────────────────────────────
# Written as a caption-ready sentence so it can be copied directly into the
# report. 'y=1.01' lifts it just above the top row of panels.
fig.suptitle(
    'Spatial Distribution of Wordplay Types in UMAP Semantic Space\n'
    'Gray: all 12,622 indicators  \u00b7  Color: indicators for each Ho wordplay '
    'type  \u00b7  Multi-label indicators appear in multiple panels',
    fontsize=11,
    y=1.01,
)

plt.tight_layout(pad=0.6, h_pad=1.2, w_pad=0.8)

out_path = REPORT_FIGURES_DIR / 'fig3_ho_type_overlay.png'
fig.savefig(out_path, dpi=FIG3_DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path.name}  ({FIG3_DPI} DPI)')
print('Panel counts:')
for wtype in HO_TYPES:
    print(f'  {wtype:12s}: {ho_type_masks[wtype].sum():5,} indicators')


### 3.2 Easy vs. Hard Separation (The ARI Contrast)

**The core quantitative finding of the project.**

Stage 5 ran two subset experiments to test whether the difficulty of separating
wordplay types is *uniform* or *type-dependent*:

| Experiment | Subset | Method | ARI vs Ho labels |
|------------|--------|--------|------------------|
| **4A** | Homophone + Reversal | Agglomerative k=2 | **0.611** |
| **4B** | Hidden + Container + Insertion | Agglomerative k=3 | **0.045** |

A ratio of **13.5×** between 4A and 4B. This is not a tuning artifact — it
reflects a real linguistic difference:

- **Homophone and reversal** indicators use highly specific, *acoustic or positional*
  language ("sounds like", "backwards", "turned"). BGE-M3 embeddings capture this
  semantic specificity, placing them in well-separated spatial regions.
- **Hidden, container, and insertion** indicators use *containment metaphors*
  ("in", "within", "holding") that are so semantically overlapping that even
  large language model embeddings cannot cleanly separate them.

**What to read from this figure:** In panel 4A, the two colors (homophone pink
and reversal blue) occupy *distinct spatial regions*. In panel 4B, the three
colors (hidden green, container purple, insertion orange) are *thoroughly
intermixed* — no spatial separation is visible, matching ARI ≈ 0.

**Implementation note:** Points are colored by `primary_ho` (the most frequent
Ho label across all clue appearances of that indicator). Multi-label indicators
may show a color outside the expected subset types — this is intentional and
reflects the genuine linguistic ambiguity of those indicators.


In [ ]:
# ─── Gather subset data ────────────────────────────────────────────
# Subset masks: use ho_type_masks (multi-label aware, built in Section 2.1)
# An indicator belongs to a subset if it appears under ANY of the listed types.
mask_4a = ho_type_masks['homophone'] | ho_type_masks['reversal']
mask_4b = (ho_type_masks['hidden']
           | ho_type_masks['container']
           | ho_type_masks['insertion'])

# UMAP 2D coordinates for each subset
x_4a = embeddings_2d[mask_4a, 0];  y_4a = embeddings_2d[mask_4a, 1]
x_4b = embeddings_2d[mask_4b, 0];  y_4b = embeddings_2d[mask_4b, 1]

# Per-indicator primary Ho label → color
# Using .values to get a numpy array so boolean indexing works cleanly
primary_4a = df_master['primary_ho'].values[mask_4a]
primary_4b = df_master['primary_ho'].values[mask_4b]
colors_4a  = [TYPE_COLORS.get(t, '#888888') for t in primary_4a]
colors_4b  = [TYPE_COLORS.get(t, '#888888') for t in primary_4b]

# ─── ARI values ──────────────────────────────────────────────────────────────
# Defaults from FINDINGS_AND_DECISIONS.md — the most important numeric result
ARI_4A, ARI_4B = 0.611, 0.045

# Try to read from the loaded subset metrics CSV (more precise than hardcoded)
_ari_col = next((c for c in ['ari_vs_ho', 'ari', 'ARI']
                 if c in df_subset_metrics.columns), None)
_exp_col = next((c for c in ['experiment', 'Experiment']
                 if c in df_subset_metrics.columns), None)
if _ari_col and _exp_col:
    for _, row in df_subset_metrics.iterrows():
        exp = str(row[_exp_col]).lower()
        ari_val = row[_ari_col]
        if 'easy' in exp and ('agglo' in exp or 'k2' in exp):
            ARI_4A = float(ari_val)
        elif 'hard' in exp and ('agglo' in exp or 'k3' in exp):
            ARI_4B = float(ari_val)

ratio = ARI_4A / ARI_4B if ARI_4B > 0 else float('inf')
print(f'4A ARI = {ARI_4A:.3f}  |  4B ARI = {ARI_4B:.3f}  |  Ratio = {ratio:.1f}\u00d7')
print(f'Subset sizes: 4A = {mask_4a.sum():,} inds | 4B = {mask_4b.sum():,} inds')
print(f'Unique primary-Ho types: 4A = {sorted(set(primary_4a))} | 4B = {sorted(set(primary_4b))}')


In [ ]:
# ─── Plot ──────────────────────────────────────────────────────────
FIG_DPI = 300
SC_KW   = dict(s=4, alpha=0.60, linewidths=0, rasterized=True)

fig4, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(14, 6))
fig4.subplots_adjust(wspace=0.18)

def _style_scatter_ax(ax):
    """Apply shared axis aesthetics: hide ticks, thin gray spines."""
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor('#bbbbbb')
        sp.set_linewidth(0.6)

def _ari_box(ax, method_str, ari_val):
    """Add a white text box in the upper-left corner showing method + ARI."""
    ax.annotate(
        f'{method_str}\nARI\u202f=\u202f{ari_val:.3f}',
        xy=(0.04, 0.96), xycoords='axes fraction',
        fontsize=9.5, va='top', ha='left', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.45', facecolor='white',
                  edgecolor='#888888', alpha=0.92),
    )

def _type_legend(ax, types_present, loc='lower right'):
    """Build a legend from the Ho types present in the panel."""
    ordered = [t for t in HO_TYPES if t in types_present]
    patches  = [mpatches.Patch(color=TYPE_COLORS[t], label=t) for t in ordered]
    # Add a catch-all for any off-subset types (multi-label spill-over)
    other_types = sorted(set(types_present) - set(ordered))
    for t in other_types:
        patches.append(mpatches.Patch(color=TYPE_COLORS.get(t, '#888'), label=f'{t}\u2020'))
    ax.legend(handles=patches, loc=loc, fontsize=8.5, framealpha=0.92,
              title='Ho label', title_fontsize=8)

# ── Left panel: Experiment 4A — easy subset (homophone + reversal) ────────────
ax_a.scatter(x_4a, y_4a, c=colors_4a, **SC_KW)
_style_scatter_ax(ax_a)
ax_a.set_xlabel('UMAP 1', fontsize=9, labelpad=2)
ax_a.set_ylabel('UMAP 2', fontsize=9, labelpad=2)
ax_a.set_title('Experiment 4A\nHomophone + Reversal',
               fontsize=11, fontweight='bold', pad=7)
_ari_box(ax_a, 'Agglomerative k=2', ARI_4A)
_type_legend(ax_a, set(primary_4a))

# ── Right panel: Experiment 4B — hard subset (hidden + container + insertion) ─
ax_b.scatter(x_4b, y_4b, c=colors_4b, **SC_KW)
_style_scatter_ax(ax_b)
ax_b.set_xlabel('UMAP 1', fontsize=9, labelpad=2)
ax_b.set_title('Experiment 4B\nHidden + Container + Insertion',
               fontsize=11, fontweight='bold', pad=7)
_ari_box(ax_b, 'Agglomerative k=3', ARI_4B)
_type_legend(ax_b, set(primary_4b))

# ── Figure-level title ────────────────────────────────────────────────────────
fig4.suptitle(
    'Conceptual Metaphor Separation (Easy vs. Hard)\n'
    f'ARI ratio: {ARI_4A:.3f} / {ARI_4B:.3f} = {ratio:.1f}\u00d7  '
    '\u2014  each point is one indicator, colored by primary Ho label',
    fontsize=12, y=1.03,
)

plt.tight_layout(pad=1.0)
out4 = REPORT_FIGURES_DIR / 'fig4_easy_vs_hard_separation.png'
fig4.savefig(out4, dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out4.name}  ({FIG_DPI} DPI)')
# † marker in legend labels indicates multi-label spill-over (off-subset primary type)


### 3.3 Constrained vs. Unconstrained Cluster Composition

**What this figure shows:** Each heatmap is a *cluster × wordplay-type composition
matrix*. Row *i* shows what fraction of cluster *i*'s members belong to each of
the 8 Ho wordplay types. A perfectly pure cluster would have a single dark cell per
row; a completely mixed cluster would have eight equally shaded cells.

**Why two panels?**
Stage 5 asked: *"Does injecting expert seed words — small sets of known indicators
per type — meaningfully improve cluster purity?"* The two panels compare:
- **Left — Unconstrained agglo k=8:** pure geometry, no label information used.
- **Right — Constrained MC7 k=7:** connectivity matrix built from 7 expert-curated
  seed groups biases the Ward's linkage toward linguistically coherent clusters.

**The finding:** Average purity rises only marginally (≈ 0.563 → ≈ 0.598), a
difference of roughly 3–4 percentage points. The visual similarity of the two
heatmaps makes this concretely legible: the structure the expert seeds reinforce
was *already present* in the BGE-M3 embedding geometry. This is evidence that
the model encodes wordplay-relevant semantic relationships without hand-holding.

**How to read purity from the heatmap:** For each row (cluster), find the darkest
cell — that fraction is the cluster's purity score. Average purity is the mean of
those maxima across all rows.

**Implementation:** Row values are proportions (sum to 1.0 per row) computed from
`ho_type_masks`, which is multi-label aware — an indicator like *"about"* is
counted in every type it belongs to, so row sums can slightly exceed 1.0 for
clusters with many multi-label indicators.


In [ ]:
# ─── Build cluster × Ho-type composition matrices ──────────────────

def build_cluster_type_matrix(labels, label_prefix='C'):
    """
    For each non-noise cluster, compute the proportion of indicators
    belonging to each Ho type.

    Returns
    -------
    pd.DataFrame  shape (n_clusters, 8)
        Rows indexed by 'C0\\n(n=X)' strings; columns = HO_TYPES.
        Values are fractions in [0, 1]. Because ho_type_masks is multi-label,
        row sums may slightly exceed 1.0.
    """
    cluster_ids = sorted(set(int(c) for c in labels if c != -1))
    rows, row_idx = [], []
    for cid in cluster_ids:
        cmask = labels == cid
        total = int(cmask.sum())
        row = {
            wtype: (ho_type_masks[wtype][cmask].sum() / total
                    if total > 0 else 0.0)
            for wtype in HO_TYPES
        }
        rows.append(row)
        row_idx.append(f'{label_prefix}{cid}\n(n={total:,})')
    return pd.DataFrame(rows, index=row_idx)[HO_TYPES]   # enforce column order


# Unconstrained: agglo k=8 (labels loaded in Section 1 as agglo_labels_dict[8])
heat_k8  = build_cluster_type_matrix(agglo_labels_dict[8],  label_prefix='C')

# Constrained: MC7 k=7 (labels loaded in Section 1 as labels_mc7)
heat_mc7 = build_cluster_type_matrix(labels_mc7, label_prefix='S')  # S = seed group

# Purity values — computed in Section 2 via compute_avg_purity()
# Fall back to FINDINGS values if Section 2 was skipped
purity_k8_val  = purity_agglo.get(8,  0.563)
purity_mc7_val = purity_mc7

print(f'Agglo k=8  avg purity: {purity_k8_val:.3f}')
print(f'MC7 k=7    avg purity: {purity_mc7_val:.3f}')
print(f'Purity gain: +{purity_mc7_val - purity_k8_val:.3f}')
print()
print('Agglo k=8 cluster composition (proportions):')
print(heat_k8.round(2).to_string())
print()
print('MC7 k=7 cluster composition (proportions):')
print(heat_mc7.round(2).to_string())


In [ ]:
# ─── Heatmaps ───────────────────────────────────────────────────────
# Shared heatmap kwargs
HEAT_KW = dict(
    cmap='Blues',
    vmin=0, vmax=1,
    annot=True, fmt='.0%',
    annot_kws={'size': 7.5},
    linewidths=0.4, linecolor='#e8e8e8',
    cbar_kws={'label': 'Proportion of cluster', 'shrink': 0.75},
)

# Figure height scales with the taller of the two heatmaps
n_rows_max = max(len(heat_k8), len(heat_mc7))
fig5_h = max(5, n_rows_max * 0.65 + 2.0)
fig5, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(16, fig5_h))
fig5.subplots_adjust(wspace=0.45)  # room for two independent colourbars

# ── Left: unconstrained agglo k=8 ────────────────────────────────────────────
sns.heatmap(heat_k8, ax=ax_l, **HEAT_KW)
ax_l.set_title(
    f'Unconstrained  Agglomerative k=8\n'
    f'(Avg Purity\u202f=\u202f{purity_k8_val:.3f})',
    fontsize=11, fontweight='bold', pad=8,
)
ax_l.set_xticklabels(HO_TYPES, rotation=35, ha='right', fontsize=9)
ax_l.set_yticklabels(ax_l.get_yticklabels(), rotation=0, fontsize=8)
ax_l.set_ylabel('Cluster  (C = cluster ID)', fontsize=9)
ax_l.set_xlabel('Ho Wordplay Type', fontsize=9)

# ── Right: constrained MC7 k=7 ───────────────────────────────────────────────
sns.heatmap(heat_mc7, ax=ax_r, **HEAT_KW)
ax_r.set_title(
    f'Constrained  MC7  k=7\n'
    f'(Avg Purity\u202f=\u202f{purity_mc7_val:.3f})',
    fontsize=11, fontweight='bold', pad=8,
)
ax_r.set_xticklabels(HO_TYPES, rotation=35, ha='right', fontsize=9)
ax_r.set_yticklabels(ax_r.get_yticklabels(), rotation=0, fontsize=8)
ax_r.set_ylabel('Cluster  (S = seed group)', fontsize=9)
ax_r.set_xlabel('Ho Wordplay Type', fontsize=9)

# ── Figure title ──────────────────────────────────────────────────────────────
purity_delta = purity_mc7_val - purity_k8_val
fig5.suptitle(
    'Cluster Composition  \u2014  Unconstrained k=8 vs. Constrained MC7\n'
    f'Cell values = proportion of each cluster belonging to each Ho type  '
    f'(purity gain: +{purity_delta:.3f})',
    fontsize=11, y=1.03,
)

plt.tight_layout(pad=1.0)
out5 = REPORT_FIGURES_DIR / 'fig5_constrained_vs_unconstrained.png'
fig5.savefig(out5, dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out5.name}  ({FIG_DPI} DPI)')


### 3.4 Anagram Sub-clustering (Appendix)

**Why anagram?** Anagram indicators make up roughly 47–51% of all indicators,
making them the dominant type by far. Early in the project we observed that even
within the anagram region of UMAP space, the embedding forms sub-clusters that
correspond to *conceptual metaphors* — the underlying cognitive framing that makes
a word feel like an anagram indicator:

| Conceptual metaphor | Example indicators |
|---------------------|--------------------|
| **Mixing / Agitation** | *stirred, blended, muddled, shaken* |
| **Disorder / Chaos** | *chaotic, jumbled, confused, wild* |
| **Change / Transformation** | *altered, converted, modified, reformed* |
| **Breaking / Destruction** | *shattered, broken, wrecked, smashed* |

This figure shows the same anagram-subset points under four different cluster
granularities, each using the *full-dataset* cluster labels (computed on all
12,622 indicators in Stage 4) but restricted here to the anagram subset only.
The four panels reveal how finer-grained clustering progressively recovers the
conceptual-metaphor sub-structure.

**What to look for:**
- **HDBSCAN** — finest granularity: many small, spatially compact clusters.
  Each cluster captures a tight semantic neighbourhood (e.g., all
  *"mix"*-family words). Noise points (gray) are indicators that HDBSCAN
  could not confidently assign.
- **Agglomerative k=4** — coarsest: 4 large blocks. At this resolution,
  distinct metaphor families are merged together.
- **Agglomerative k=8 / k≈12** — intermediate: the spatial boundaries become
  increasingly fine, matching the conceptual-metaphor taxonomy more closely.

**Implementation note:** Colors cycle through a qualitative colormap (tab20 for
HDBSCAN, Set1/tab10 for smaller k). Because the HDBSCAN panel may have more
clusters than the 20 available tab20 colors, colors repeat — but since each
cluster occupies a distinct spatial region, visual confusion is minimal.


In [ ]:
# ─── Prepare anagram subset and resolve k≈12 ───────────────────────

# Anagram mask: ALL indicators that appear under 'anagram' in at least one clue
anagram_mask = ho_type_masks['anagram']
x_anag = embeddings_2d[anagram_mask, 0]
y_anag = embeddings_2d[anagram_mask, 1]
n_anag = int(anagram_mask.sum())
print(f'Anagram subset: {n_anag:,} of {len(indicator_names):,} indicators '
      f'({n_anag / len(indicator_names):.1%})')

# ── Resolve k≈12 ─────────────────────────────────────────────────────────────
# The exact k values available depend on what Stage 4 computed.
# Try k=12 first; if absent, fall back to the nearest available k.
_target_k = 12
_k12_actual = None
for k_try in [12, 10, 14, 11, 13, 15, 16]:
    if k_try in agglo_labels_dict:
        _k12_actual = k_try
        break

if _k12_actual is None:
    # Last resort: use the smallest k larger than 8
    candidates = [k for k in sorted(agglo_labels_dict) if k > 8]
    _k12_actual = candidates[0] if candidates else 8

k12_panel_title = f'Agglomerative k={_k12_actual}'
if _k12_actual != _target_k:
    k12_panel_title += f'\n(k={_target_k} not computed; using k={_k12_actual})'

# ── Panel definitions ─────────────────────────────────────────────────────────
# Each entry: (title, full-dataset label array, colormap name)
# The label arrays span all 12,622 indicators; we index with anagram_mask below.
_eps0 = min(hdbscan_labels_dict.keys())    # eps=0.0 → finest HDBSCAN granularity
FIG6_PANELS = [
    (f'HDBSCAN  \u03b5={_eps0:.2f}',    hdbscan_labels_dict[_eps0],         'tab20'),
    ('Agglomerative k=4',                agglo_labels_dict.get(4),           'Set1'),
    ('Agglomerative k=8',                agglo_labels_dict.get(8),           'tab10'),
    (k12_panel_title,                    agglo_labels_dict.get(_k12_actual), 'tab20'),
]

print('\nPanel summary (anagram subset only):')
for title, full_lbl, _ in FIG6_PANELS:
    short = title.split('\n')[0]
    if full_lbl is None:
        print(f'  {short}: NOT AVAILABLE')
    else:
        sub = full_lbl[anagram_mask]
        n_cl = len(set(int(l) for l in sub if l != -1))
        n_no = int((sub == -1).sum())
        print(f'  {short}: {n_cl} clusters, {n_no} noise points')


In [ ]:
# ─── 2×2 anagram sub-clustering grid ────────────────────────────────
import matplotlib

def _discrete_cmap(name, n):
    """Return a colormap sampled at n discrete, evenly-spaced colours.
    Compatible with both matplotlib ≥3.7 (colormaps attribute) and older."""
    try:
        return matplotlib.colormaps[name].resampled(max(n, 1))
    except AttributeError:
        return plt.cm.get_cmap(name, max(n, 1))   # deprecated but functional

fig6, axes6 = plt.subplots(2, 2, figsize=(14, 10))
axes6_flat  = axes6.flatten()

for ax, (panel_title, full_labels, cmap_name) in zip(axes6_flat, FIG6_PANELS):

    if full_labels is None:
        # Graceful fallback if Stage 4 didn't produce this k value
        ax.text(0.5, 0.5, 'Labels not available\n(re-run 04_clustering.ipynb)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=10, color='#888888')
        ax.set_title(panel_title.split('\n')[0], fontsize=10.5, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
        continue

    # Restrict to anagram subset
    sub_labels     = full_labels[anagram_mask]
    unique_clusters = sorted(set(int(l) for l in sub_labels if l != -1))
    n_clusters_here = len(unique_clusters)

    # Build a cluster-id → RGBA colour mapping.
    # For HDBSCAN with many clusters, tab20 cycles (20 colours); for small k,
    # Set1/tab10 give distinct, non-repeating colours.
    cmap_obj    = _discrete_cmap(cmap_name, max(n_clusters_here, 1))
    id_to_color = {
        cid: cmap_obj(i % cmap_obj.N)
        for i, cid in enumerate(unique_clusters)
    }

    # Layer 1 — noise (HDBSCAN only): light gray, very small, very transparent
    noise_mask = sub_labels == -1
    if noise_mask.sum() > 0:
        ax.scatter(
            x_anag[noise_mask], y_anag[noise_mask],
            c='#d4d4d4', s=0.8, alpha=0.25, linewidths=0, rasterized=True,
            zorder=1,
        )

    # Layer 2 — cluster points: each cluster is its own scatter call so
    # matplotlib assigns a distinct colour from id_to_color
    for cid in unique_clusters:
        cmask = sub_labels == cid
        ax.scatter(
            x_anag[cmask], y_anag[cmask],
            color=id_to_color[cid],
            s=2.5, alpha=0.55, linewidths=0, rasterized=True,
            zorder=2,
        )

    # Cluster + noise count annotation box (lower left)
    noise_note = f'  |  {noise_mask.sum():,} noise' if noise_mask.sum() > 0 else ''
    ax.annotate(
        f'{n_clusters_here} clusters{noise_note}',
        xy=(0.04, 0.04), xycoords='axes fraction',
        fontsize=8, va='bottom', ha='left',
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                  edgecolor='#aaaaaa', alpha=0.88),
    )

    ax.set_title(panel_title, fontsize=10.5, fontweight='bold', pad=6)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor('#bbbbbb'); sp.set_linewidth(0.6)

# ── Figure title ──────────────────────────────────────────────────────────────
fig6.suptitle(
    f'Anagram Sub-clustering by Conceptual Metaphor\n'
    f'Showing only the {n_anag:,} anagram-type indicators '
    f'({n_anag / len(indicator_names):.0%} of all indicators)  '
    f'\u2014  same 2D UMAP embedding for all panels',
    fontsize=12, y=1.02,
)

plt.tight_layout(pad=1.0, h_pad=1.5, w_pad=1.0)
out6 = REPORT_FIGURES_DIR / 'fig6_anagram_subclustering.png'
fig6.savefig(out6, dpi=FIG_DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out6.name}  ({FIG_DPI} DPI)')


---
## Section 4: Summary Table and Report Figure Inventory

The previous sections produced individual figures and computed per-run metrics.
Section 4 does two things:

1. **Part 1 — Master Summary Table:** Aggregates *all* quantitative results from
   Stages 4 and 5 into a single authoritative DataFrame. This is the table that
   will appear in the report appendix and serves as the single source of truth
   for every metric claim made in the paper.

2. **Part 2 — Report Figure Inventory:** A structured list that maps every
   figure (generated in NB 04, NB 05, or NB 06) to the specific MADS Capstone
   rubric requirement it satisfies. Use this inventory when assembling the final
   report to ensure no rubric item is left uncovered.


### 4.1 Master Summary Table

All quantitative metrics produced across Stages 4 and 5 are joined here into
a single DataFrame with a consistent column schema:

| Column | Source | Notes |
|--------|--------|-------|
| **Method** | — | Human-readable method family label |
| **Config** | NB04/05 CSVs | ε value (HDBSCAN) or k (agglomerative) |
| **Clusters** | NB04/05 CSVs | Actual non-noise clusters found or specified |
| **Silhouette** | `clustering_metrics_summary.csv` | Higher is better; range −1 to 1 |
| **Davies-Bouldin** | `clustering_metrics_summary.csv` | Lower is better; ≥ 0 |
| **Calinski-Harabasz** | `clustering_metrics_summary.csv` | Higher is better; may be NaN for HDBSCAN with noise |
| **Avg Purity** | Computed in §2 / `constrained_vs_unconstrained_metrics.csv` | Mean dominant-type fraction per cluster |
| **Noise %** | `clustering_metrics_summary.csv` | Non-zero only for HDBSCAN |
| **ARI vs Ho** | `subset_experiment_metrics.csv` | Non-null only for subset experiments |

**Missing values (`—`)** appear where a metric is not applicable to a run type:
agglomerative clustering produces no noise points so Noise % = 0; Calinski-Harabasz
is undefined when noise points are excluded from label arrays; ARI is only
meaningful for the Stage 5 subset experiments that have a ground-truth reference.

The table is exported to `outputs/final_metrics_summary.csv` for inclusion in
the report appendix.


In [ ]:
from IPython.display import display

# ─── Helper: tolerant purity look-up for HDBSCAN ─────────────────────────────
# df_metrics stores epsilon as a float from the CSV; purity_hdbscan keys come
# from the label-dict keys (parsed from filenames). A tiny tolerance avoids
# floating-point mismatches between the two sources.
def _hdbscan_purity(eps, tol=1e-5):
    for k in purity_hdbscan:
        if abs(k - eps) < tol:
            return purity_hdbscan[k]
    return np.nan


# ─── Part A: NB04 runs (HDBSCAN sweep + Agglomerative sweep) ─────────────────
rows_nb04 = []
for _, row in df_metrics.iterrows():
    is_hdb = str(row.get('method', '')).lower() == 'hdbscan'

    if is_hdb:
        eps       = float(row.get('epsilon', np.nan))
        n_cl      = row.get('n_clusters', np.nan)
        noise_pct = float(row.get('noise_pct', np.nan))
        config    = f'\u03b5={eps:.4f}'
        purity    = _hdbscan_purity(eps) if not np.isnan(eps) else np.nan
        method    = 'HDBSCAN'
    else:
        k_val     = row.get('k', np.nan)
        n_cl      = k_val
        noise_pct = 0.0
        config    = f'k={int(k_val)}' if not pd.isna(k_val) else '—'
        purity    = purity_agglo.get(int(k_val), np.nan) if not pd.isna(k_val) else np.nan
        method    = "Agglomerative (Ward's)"

    rows_nb04.append({
        'Method':            method,
        'Config':            config,
        'Clusters':          (int(n_cl) if not pd.isna(n_cl) else np.nan),
        'Silhouette':        row.get('silhouette',         np.nan),
        'Davies-Bouldin':    row.get('davies_bouldin',     np.nan),
        'Calinski-Harabasz': row.get('calinski_harabasz',  np.nan),
        'Avg Purity':        purity,
        'Noise %':           noise_pct,
        'ARI vs Ho':         np.nan,
    })

# ─── Part B: NB05 constrained runs ───────────────────────────────────────────
rows_constrained = []
_run_col = next((c for c in ['run', 'Run', 'method'] if c in df_constrained.columns), None)
_k_col   = next((c for c in ['k', 'K', 'n_clusters'] if c in df_constrained.columns), None)
if _run_col:
    for _, row in df_constrained.iterrows():
        k_val = row.get(_k_col, np.nan) if _k_col else np.nan
        rows_constrained.append({
            'Method':            'Constrained Agglomerative',
            'Config':            str(row[_run_col]),
            'Clusters':          (int(k_val) if not pd.isna(k_val) else np.nan),
            'Silhouette':        row.get('silhouette',     np.nan),
            'Davies-Bouldin':    row.get('davies_bouldin', np.nan),
            'Calinski-Harabasz': np.nan,          # not computed for constrained runs
            'Avg Purity':        row.get('avg_purity',    np.nan),
            'Noise %':           0.0,
            'ARI vs Ho':         np.nan,
        })

# ─── Part C: NB05 subset experiment runs ─────────────────────────────────────
# These runs produced ARI scores against Ho labels; other metrics may be partial.
rows_subset = []
_exp_col = next((c for c in ['experiment', 'Experiment'] if c in df_subset_metrics.columns), None)
_ari_col = next((c for c in ['ari_vs_ho', 'ari', 'ARI']  if c in df_subset_metrics.columns), None)
if _exp_col:
    for _, row in df_subset_metrics.iterrows():
        k_val = row.get('k', np.nan)
        rows_subset.append({
            'Method':            'Subset Experiment',
            'Config':            str(row[_exp_col]),
            'Clusters':          (int(k_val) if not pd.isna(k_val) else np.nan),
            'Silhouette':        row.get('silhouette', np.nan),
            'Davies-Bouldin':    np.nan,
            'Calinski-Harabasz': np.nan,
            'Avg Purity':        np.nan,
            'Noise %':           np.nan,
            'ARI vs Ho':         row.get(_ari_col, np.nan) if _ari_col else np.nan,
        })

# ─── Assemble and sort ────────────────────────────────────────────────────────
df_summary = pd.DataFrame(rows_nb04 + rows_constrained + rows_subset)

# Sort: method family first, then best silhouette at top within each family
_method_rank = {
    'HDBSCAN': 0, "Agglomerative (Ward's)": 1,
    'Constrained Agglomerative': 2, 'Subset Experiment': 3,
}
df_summary['_rank'] = df_summary['Method'].map(_method_rank)
df_summary = (
    df_summary
    .sort_values(['_rank', 'Silhouette'], ascending=[True, False])
    .drop(columns='_rank')
    .reset_index(drop=True)
)

# Round all float metrics to 3 decimal places (except Calinski-Harabasz → 1)
for col in ['Silhouette', 'Davies-Bouldin', 'Avg Purity', 'Noise %', 'ARI vs Ho']:
    df_summary[col] = df_summary[col].round(3)
df_summary['Calinski-Harabasz'] = df_summary['Calinski-Harabasz'].round(1)

# ─── Export to CSV ────────────────────────────────────────────────────────────
# Saved directly into outputs/ (the 'outputs/outputs/' path in the prompt
# was a typo — OUTPUT_DIR already points to the outputs/ directory).
out_csv = OUTPUT_DIR / 'final_metrics_summary.csv'
df_summary.to_csv(out_csv, index=False)
print(f'Exported {len(df_summary)} rows \u00d7 {len(df_summary.columns)} columns')
print(f'  \u2192 {out_csv}')
print(f'  Breakdown: {dict(df_summary["Method"].value_counts())}')
print()

# ─── Styled display ───────────────────────────────────────────────────────────
# background_gradient on Silhouette (green = high) and Avg Purity (blue = high)
# highlight_min on Davies-Bouldin (green = low = better)
# NaN shown as an em-dash for readability
_COLS = ['Method', 'Config', 'Clusters',
         'Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz',
         'Avg Purity', 'Noise %', 'ARI vs Ho']
_FMT  = {
    'Silhouette':        '{:.3f}',
    'Davies-Bouldin':    '{:.3f}',
    'Calinski-Harabasz': '{:.1f}',
    'Avg Purity':        '{:.3f}',
    'Noise %':           '{:.1f}',
    'ARI vs Ho':         '{:.3f}',
}

styled = (
    df_summary[_COLS].style
    .format(_FMT, na_rep='\u2014')          # '—' for NaN
    .background_gradient(
        subset=['Silhouette'], cmap='Greens',
        vmin=0.0, vmax=df_summary['Silhouette'].max(),
    )
    .background_gradient(
        subset=['Avg Purity'], cmap='Blues',
        vmin=0.0, vmax=1.0,
    )
    .set_properties(**{
        'font-size': '11px',
        'text-align': 'center',
        'white-space': 'nowrap',
    })
    .set_properties(subset=['Method', 'Config'], **{'text-align': 'left'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('font-size', '11px'), ('text-align', 'center'),
                  ('background-color', '#f5f5f5')],
    }])
    .set_caption(
        'Master Metrics Summary — all Stage 4 and Stage 5 runs.  '
        'Green = high silhouette; Blue = high purity; \u2014 = not applicable.'
    )
)

display(styled)


### 4.2 Report Figure Inventory

The table below maps every figure in the project to the MADS Capstone rubric
requirement it satisfies. Rubric items addressed:

- **R1** — At least 2 visualizations per method family
- **R2** — At least one sensitivity analysis
- **R3** — At least one overall summary comparing the best model from each family
- **R4** — Justification of chosen metrics with reference to data properties
- **R5** — Evidence for or against the research question

---

#### Main Report — Agglomerative Clustering  `[R1, R5]`

| # | Figure | Source | Rubric | What it proves |
|---|--------|--------|--------|----------------|
| — | **Truncated Dendrogram** | NB 04 | R1, R4 | No natural cut at k=8; hierarchy is continuous. Justifies why we sweep k rather than target k=8 directly. |
| 4 | **Easy vs. Hard Separation** (`fig4_easy_vs_hard_separation.png`) | NB 06 §3.2 | R1, R5 | The 13.5× ARI ratio (0.611 vs. 0.045) is the single strongest quantitative result. Homophone/Reversal points are spatially separated; Hidden/Container/Insertion points are thoroughly intermixed. |

---

#### Main Report — HDBSCAN  `[R1, R2]`

| # | Figure | Source | Rubric | What it proves |
|---|--------|--------|--------|----------------|
| — | **Epsilon Sensitivity Sweep** | NB 04 | R2 | Primary sensitivity analysis: clusters found and silhouette score vs ε. Proves the abrupt transition — no stable intermediate regime exists. |
| 2 | **Sensitivity Analysis** (`fig2_sensitivity_analysis.png`) | NB 06 §2.3 | R2 | Report-quality version of the sensitivity analysis. Left panel (HDBSCAN) and right panel (Agglomerative) side-by-side confirm monotonic behavior in both method families. |
| 3 | **Ho Type Overlay** (`fig3_ho_type_overlay.png`) | NB 06 §3.1 | R1, R5 | Spatial footprint of all 8 types. Shows homophone occupies a compact, isolated UMAP region (supports HDBSCAN finding it at eps=0), while anagram is diffuse and multi-modal. |

---

#### Main Report — Method Comparison  `[R3, R4, R5]`

| # | Figure | Source | Rubric | What it proves |
|---|--------|--------|--------|----------------|
| 1 | **Unified Metrics Comparison** (`fig1_unified_metrics_comparison.png`) | NB 06 §2.2 | R3, R4 | All 32 runs (13 HDBSCAN ε + 17 Agglomerative k + 2 Constrained) on three metrics side-by-side. The definitive cross-family comparison table. |
| 5 | **Constrained vs. Unconstrained Heatmaps** (`fig5_constrained_vs_unconstrained.png`) | NB 06 §3.3 | R3, R5 | Per-cluster Ho-type composition for agglo k=8 (unconstrained) vs. MC7 k=7 (constrained). Marginal purity gain (+0.03–0.04) proves BGE-M3 embeddings already encode wordplay-relevant semantic organization without expert seed-word guidance. |

---

#### Appendix — Supporting Detail

| # | Figure | Source | Rubric | What it provides |
|---|--------|--------|--------|------------------|
| 6 | **Anagram Sub-clustering** (`fig6_anagram_subclustering.png`) | NB 06 §3.4 | R5 | Four-panel view of HDBSCAN, k=4, k=8, and k≈12 on the anagram-only subset. Proves conceptual-metaphor hierarchy within the dominant type (Mixing, Disorder, Change, Breaking themes). |
| — | **k=10 Scatter Plot** | NB 04 | R1 | Shows the local silhouette optimum at k=10: homophone (0.78 purity) and reversal (0.90) recovered as near-pure clusters. |
| — | **k=34 Scatter Plot** | NB 04 | R1 | Fine-grained agglomerative solution; one cluster per CG34 conceptual group. Demonstrates that higher k monotonically improves purity at the cost of interpretability. |

---

#### Coverage Checklist

| Rubric Requirement | Satisfied by | Status |
|--------------------|-------------|--------|
| R1: ≥2 viz per method (HDBSCAN) | Figs 2, 3 + NB04 epsilon sweep | ✅ |
| R1: ≥2 viz per method (Agglomerative) | Fig 4 + NB04 dendrogram | ✅ |
| R2: Sensitivity analysis | Fig 2 + NB04 epsilon sweep | ✅ |
| R3: Cross-family comparison | Fig 1 + Fig 5 | ✅ |
| R4: Metric justification | Fig 1 annotations + §4.1 table | ✅ |
| R5: Research question evidence | Figs 3, 4, 5, 6 | ✅ |
